# 01 - Decide, then measure the baseline

Most tasks should **not** be fine-tuned. Before training anything, prove the
baseline is not good enough. This notebook builds the eval set and measures a
prompting baseline against a real model.

**Decision tree** (from the S7 handout):
1. Build a 100+ eval set, measure the base model with a good prompt.
2. Try prompt + few-shot. If it closes 80% of the gap, stop - no fine-tune.
3. If the failure is factual, try RAG. If that closes it, stop.
4. Only now fine-tune, and re-measure on the **same** eval set.
5. If fine-tuning does not gain 10%+, your fine-tune is wrong, not the model.

In [ ]:
import sys; sys.path.insert(0, '../src')
from qlora_lab import schema, synth, dataset, predict, evaluate, train, serve, agent

### Point the client at any OpenAI-compatible model
Use a frontier API as the 'can it be done at all' ceiling, or a small base
model served locally by vLLM as the honest baseline you will try to beat.

In [ ]:
import os
from openai import OpenAI
# frontier baseline:
client = OpenAI()  # reads OPENAI_API_KEY
MODEL = 'gpt-4o-mini'
# local small base instead:
# client = serve.openai_client('http://localhost:8000/v1')
# MODEL = 'unsloth/Qwen3-8B-bnb-4bit'

In [ ]:
test = dataset.read_jsonl('../data/test.jsonl')  # from scripts/make_data.py
preds = [predict.extract(client, MODEL, e['message']) for e in test]
rep = evaluate.evaluate(preds, test, in_price=0.15e-6, out_price=0.60e-6)
print(rep.summary())
print('first failures:', [f['reason'] for f in rep.failures[:5]])

Save this baseline. The fine-tune in notebook 03 has to beat it on the same
100 test examples or it is not worth shipping.